# Phase 1-2: CAUEEG 이벤트 구조 분석

**목표**: 녹음 내 이벤트 시퀀스 패턴 파악 → EDCC 토크나이징 전략 수립

**분석 내용**:
1. 이벤트 유형별 빈도 집계
2. Eyes-Open / Eyes-Closed 구간 길이 분포 (class별)
3. 이벤트 시퀀스 패턴 유형화
4. 이벤트 간 전환 빈도 매트릭스
5. 녹음 프로토콜 표준 패턴 도출
6. EDCC 이벤트 세그먼테이션 전략 검증

In [ ]:
import sys, os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict

sys.path.insert(0, os.path.abspath('..'))

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

DATASET_PATH = '../local/dataset/caueeg-dataset'
SAMPLING_RATE = 200

In [ ]:
from datasets.caueeg_script import load_caueeg_task_datasets

task_config, train_ds, val_ds, test_ds = load_caueeg_task_datasets(
    DATASET_PATH, task='dementia', load_event=True, file_format='edf'
)

class_names = task_config['class_label_to_name']
all_datasets = {'train': train_ds, 'val': val_ds, 'test': test_ds}

# 전체 이벤트 데이터 수집
event_records = []
for split_name, ds in all_datasets.items():
    for i in range(len(ds)):
        sample = ds[i]
        events = sample.get('event', [])
        signal_length = sample['signal'].shape[1]
        
        event_records.append({
            'serial': sample['serial'],
            'split': split_name,
            'class_label': sample['class_label'],
            'class_name': class_names[sample['class_label']],
            'age': sample['age'],
            'signal_length': signal_length,
            'events': events,
            'n_events': len(events),
        })

df_events = pd.DataFrame(event_records)
print(f"Total recordings: {len(df_events)}")
print(f"Events per recording: mean={df_events['n_events'].mean():.1f}, "
      f"std={df_events['n_events'].std():.1f}, "
      f"min={df_events['n_events'].min()}, max={df_events['n_events'].max()}")

## 1. 이벤트 유형별 빈도 집계

In [ ]:
# 이벤트 유형별 빈도
def normalize_event_type(event_name):
    """이벤트 이름을 정규화된 카테고리로 변환"""
    event_name = event_name.strip()
    if event_name == 'Eyes Open':
        return 'Eyes Open'
    elif event_name == 'Eyes Closed':
        return 'Eyes Closed'
    elif event_name.startswith('Photic On'):
        return 'Photic On'
    elif event_name == 'Photic Off':
        return 'Photic Off'
    elif event_name in ('Start Recording', 'Recording Resumed'):
        return 'Recording Control'
    elif event_name == 'Paused':
        return 'Paused'
    elif event_name.startswith('New Montage'):
        return 'Montage'
    elif event_name in ('Move', 'swallowing', 'artifact'):
        return 'Artifact'
    else:
        return 'Other'

# 전체 이벤트 수집
all_event_types = Counter()
all_event_names = Counter()
per_recording_event_counts = []

for _, row in df_events.iterrows():
    rec_counts = Counter()
    for sample_idx, event_name in row['events']:
        all_event_names[event_name] += 1
        norm_type = normalize_event_type(event_name)
        all_event_types[norm_type] += 1
        rec_counts[norm_type] += 1
    per_recording_event_counts.append(rec_counts)

# 이벤트 유형별 빈도
print("=== 이벤트 유형별 전체 빈도 ===")
for evt, count in all_event_types.most_common():
    print(f"  {evt:25s}: {count:6d}")

print(f"\n=== 원본 이벤트 이름 (상세) ===")
for evt, count in all_event_names.most_common():
    print(f"  {evt:35s}: {count:6d}")

In [ ]:
# 녹음당 EO/EC 이벤트 수 분포
df_events['n_eo'] = [c.get('Eyes Open', 0) for c in per_recording_event_counts]
df_events['n_ec'] = [c.get('Eyes Closed', 0) for c in per_recording_event_counts]
df_events['n_photic'] = [c.get('Photic On', 0) for c in per_recording_event_counts]
df_events['has_eo_ec'] = (df_events['n_eo'] > 0) & (df_events['n_ec'] > 0)
df_events['has_photic'] = df_events['n_photic'] > 0

print(f"EO/EC 이벤트가 있는 녹음: {df_events['has_eo_ec'].sum()} / {len(df_events)} "
      f"({df_events['has_eo_ec'].mean()*100:.1f}%)")
print(f"Photic 이벤트가 있는 녹음: {df_events['has_photic'].sum()} / {len(df_events)} "
      f"({df_events['has_photic'].mean()*100:.1f}%)")

# Class별 분포
print("\n=== Class별 EO/EC 존재 비율 ===")
for cls in ['Normal', 'MCI', 'Dementia']:
    subset = df_events[df_events['class_name'] == cls]
    print(f"  {cls}: EO/EC={subset['has_eo_ec'].mean()*100:.1f}%, "
          f"Photic={subset['has_photic'].mean()*100:.1f}%, "
          f"avg EO={subset['n_eo'].mean():.1f}, avg EC={subset['n_ec'].mean():.1f}")

## 2. EO/EC 구간 길이 분포

각 Eyes-Open과 Eyes-Closed 구간의 길이를 분석합니다.
EDCC의 2초 윈도우 크기가 적절한지 확인합니다.

In [ ]:
# EO/EC 구간 추출 및 길이 계산
segment_records = []

for _, row in df_events.iterrows():
    events = row['events']
    signal_length = row['signal_length']
    
    # 관심 이벤트만 필터링 (EO, EC)
    eo_ec_events = []
    for sample_idx, event_name in events:
        if event_name in ('Eyes Open', 'Eyes Closed'):
            eo_ec_events.append((sample_idx, event_name))
    
    # 연속된 이벤트 간 구간 계산
    for i in range(len(eo_ec_events)):
        start_idx, start_type = eo_ec_events[i]
        
        # 다음 EO/EC 이벤트까지의 구간 또는 녹음 끝
        if i + 1 < len(eo_ec_events):
            end_idx = eo_ec_events[i + 1][0]
        else:
            end_idx = signal_length
        
        duration_sec = (end_idx - start_idx) / SAMPLING_RATE
        
        # 비정상적으로 긴 구간 (2분 이상) 또는 짧은 구간 (0.5초 미만) 제외하지 않고 기록
        segment_records.append({
            'serial': row['serial'],
            'class_name': row['class_name'],
            'segment_type': start_type,
            'start_sample': start_idx,
            'end_sample': end_idx,
            'duration_sec': duration_sec,
        })

df_segments = pd.DataFrame(segment_records)
print(f"Total EO/EC segments: {len(df_segments)}")
print(f"  Eyes Open: {len(df_segments[df_segments['segment_type'] == 'Eyes Open'])}")
print(f"  Eyes Closed: {len(df_segments[df_segments['segment_type'] == 'Eyes Closed'])}")

# 구간 길이 통계
print("\n=== 구간 길이 통계 (초) ===")
for seg_type in ['Eyes Open', 'Eyes Closed']:
    subset = df_segments[df_segments['segment_type'] == seg_type]['duration_sec']
    print(f"\n{seg_type}:")
    print(f"  mean={subset.mean():.2f}s, std={subset.std():.2f}s, "
          f"median={subset.median():.2f}s")
    print(f"  min={subset.min():.2f}s, max={subset.max():.2f}s")
    print(f"  < 2초 (윈도우 크기): {(subset < 2).sum()} ({(subset < 2).mean()*100:.1f}%)")
    print(f"  < 1초: {(subset < 1).sum()} ({(subset < 1).mean()*100:.1f}%)")

In [ ]:
# 구간 길이 분포 시각화
colors = {'Normal': '#2ecc71', 'MCI': '#f39c12', 'Dementia': '#e74c3c'}
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for seg_idx, seg_type in enumerate(['Eyes Open', 'Eyes Closed']):
    for cls in ['Normal', 'MCI', 'Dementia']:
        subset = df_segments[(df_segments['segment_type'] == seg_type) & 
                             (df_segments['class_name'] == cls)]['duration_sec']
        # 60초 이하만 시각화
        subset = subset[subset <= 60]
        axes[seg_idx].hist(subset, bins=50, alpha=0.5, label=cls, color=colors[cls], density=True)
    
    axes[seg_idx].axvline(x=2.0, color='red', linestyle='--', label='Window size (2s)')
    axes[seg_idx].set_xlabel('Duration (seconds)')
    axes[seg_idx].set_ylabel('Density')
    axes[seg_idx].set_title(f'{seg_type} Segment Duration')
    axes[seg_idx].legend()

plt.tight_layout()
plt.show()

## 3. 이벤트 전환 빈도 매트릭스

어떤 이벤트 다음에 어떤 이벤트가 오는지 패턴을 분석합니다.

In [ ]:
# 이벤트 전환 매트릭스
transition_counts = defaultdict(lambda: defaultdict(int))

for _, row in df_events.iterrows():
    events = row['events']
    for i in range(len(events) - 1):
        from_type = normalize_event_type(events[i][1])
        to_type = normalize_event_type(events[i + 1][1])
        transition_counts[from_type][to_type] += 1

# 주요 이벤트만 필터링
key_events = ['Eyes Open', 'Eyes Closed', 'Photic On', 'Photic Off', 'Artifact']
transition_matrix = pd.DataFrame(0, index=key_events, columns=key_events)

for from_evt in key_events:
    for to_evt in key_events:
        transition_matrix.loc[from_evt, to_evt] = transition_counts[from_evt][to_evt]

# 시각화
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(transition_matrix, annot=True, fmt='d', cmap='YlOrRd', ax=ax)
ax.set_title('Event Transition Matrix (주요 이벤트)')
ax.set_xlabel('To Event')
ax.set_ylabel('From Event')
plt.tight_layout()
plt.show()

# 주요 전환 패턴
print("\n주요 전환 패턴:")
print(f"  EO → EC: {transition_counts['Eyes Open']['Eyes Closed']}")
print(f"  EC → EO: {transition_counts['Eyes Closed']['Eyes Open']}")
print(f"  EO → Photic: {transition_counts['Eyes Open']['Photic On']}")
print(f"  EC → Photic: {transition_counts['Eyes Closed']['Photic On']}")
print(f"  Photic Off → EO: {transition_counts['Photic Off']['Eyes Open']}")
print(f"  Photic Off → EC: {transition_counts['Photic Off']['Eyes Closed']}")

## 4. 녹음 프로토콜 패턴 분석

전형적인 녹음 시퀀스 패턴을 파악합니다.

In [ ]:
# 녹음 프로토콜 시퀀스 패턴화
def get_protocol_sequence(events):
    """이벤트 리스트에서 프로토콜 시퀀스를 추출"""
    seq = []
    for _, event_name in events:
        norm = normalize_event_type(event_name)
        if norm in ('Eyes Open', 'Eyes Closed', 'Photic On', 'Photic Off'):
            # 연속 중복 제거
            if not seq or seq[-1] != norm:
                seq.append(norm)
    return seq

# 각 녹음의 프로토콜 시퀀스 분석
protocol_phases = []  # [pre_photic_eo_ec, photic, post_photic_eo_ec]

for _, row in df_events.iterrows():
    seq = get_protocol_sequence(row['events'])
    
    # Photic 전/후 구분
    has_photic = 'Photic On' in seq
    
    # EO/EC 전환 횟수 (photic 제외)
    eo_ec_transitions = sum(1 for i in range(len(seq)-1) 
                           if (seq[i] == 'Eyes Open' and seq[i+1] == 'Eyes Closed') or
                              (seq[i] == 'Eyes Closed' and seq[i+1] == 'Eyes Open'))
    
    protocol_phases.append({
        'serial': row['serial'],
        'class_name': row['class_name'],
        'has_photic': has_photic,
        'n_eo_ec_transitions': eo_ec_transitions,
        'total_events_in_seq': len(seq),
        'sequence_summary': ' → '.join(seq[:10]) + ('...' if len(seq) > 10 else ''),
    })

df_protocol = pd.DataFrame(protocol_phases)

# 프로토콜 통계
print("=== 프로토콜 통계 ===")
print(f"Photic 포함 녹음: {df_protocol['has_photic'].sum()} / {len(df_protocol)}")
print(f"EO↔EC 전환 횟수: mean={df_protocol['n_eo_ec_transitions'].mean():.1f}, "
      f"std={df_protocol['n_eo_ec_transitions'].std():.1f}")

# 전환 횟수 분포
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df_protocol['n_eo_ec_transitions'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Number of EO↔EC transitions')
axes[0].set_ylabel('Count')
axes[0].set_title('EO↔EC Transition Count per Recording')

# class별 전환 횟수
sns.boxplot(data=df_protocol, x='class_name', y='n_eo_ec_transitions',
            order=['Normal', 'MCI', 'Dementia'], palette=colors, ax=axes[1])
axes[1].set_title('EO↔EC Transitions by Class')
axes[1].set_ylabel('Number of Transitions')

plt.tight_layout()
plt.show()

# 전형적 시퀀스 예시
print("\n=== 전형적 프로토콜 시퀀스 (첫 5개) ===")
for _, row in df_protocol.head(5).iterrows():
    print(f"  [{row['class_name']:8s}] {row['sequence_summary']}")

## 5. 녹음 타임라인 시각화

샘플 녹음의 이벤트 타임라인을 시각화하여 녹음 프로토콜을 이해합니다.

In [ ]:
# 이벤트 타임라인 시각화 (class별 3개씩)
fig, axes = plt.subplots(3, 1, figsize=(16, 9), sharex=False)

event_colors_map = {
    'Eyes Open': '#2ecc71',
    'Eyes Closed': '#3498db',
    'Photic On': '#e74c3c',
    'Photic Off': '#95a5a6',
    'Artifact': '#f39c12',
    'Recording Control': '#bdc3c7',
    'Montage': '#bdc3c7',
    'Paused': '#7f8c8d',
    'Other': '#bdc3c7',
}

for ax_idx, target_class in enumerate(['Normal', 'MCI', 'Dementia']):
    subset = df_events[df_events['class_name'] == target_class].iloc[0]
    events = subset['events']
    total_sec = subset['signal_length'] / SAMPLING_RATE
    
    ax = axes[ax_idx]
    
    # 각 이벤트를 타임라인에 표시
    current_state = None
    current_start = 0
    
    for i, (sample_idx, event_name) in enumerate(events):
        norm_type = normalize_event_type(event_name)
        time_sec = sample_idx / SAMPLING_RATE
        
        # 이전 구간 색칠
        if current_state and current_state in event_colors_map:
            ax.barh(0, time_sec - current_start, left=current_start, height=0.6,
                    color=event_colors_map[current_state], alpha=0.7)
        
        current_state = norm_type
        current_start = time_sec
    
    # 마지막 구간
    if current_state and current_state in event_colors_map:
        ax.barh(0, total_sec - current_start, left=current_start, height=0.6,
                color=event_colors_map[current_state], alpha=0.7)
    
    ax.set_xlim(0, total_sec)
    ax.set_yticks([])
    ax.set_title(f'{target_class} (serial={subset["serial"]}, {total_sec:.0f}s)',
                 fontweight='bold')
    ax.set_xlabel('Time (seconds)')

# 범례
from matplotlib.patches import Patch
legend_items = [Patch(facecolor=c, label=l) 
                for l, c in event_colors_map.items() 
                if l in ('Eyes Open', 'Eyes Closed', 'Photic On', 'Artifact')]
fig.legend(handles=legend_items, loc='lower center', ncol=4, fontsize=11)

plt.suptitle('Recording Event Timeline (sample recordings)', fontsize=14, fontweight='bold')
plt.tight_layout(rect=[0, 0.08, 1, 0.95])
plt.show()

## 6. EDCC 이벤트 세그먼테이션 전략 검증

EDCC에서 사용할 이벤트 세그먼트 타입별 예상 윈도우 수를 계산합니다.
- EYES_OPEN, EYES_CLOSED: 정적 구간
- EO_TO_EC, EC_TO_EO: 전환 구간 (±3초)
- PHOTIC: Photic stimulation 구간
- OTHER: 분류 불가 구간

In [ ]:
# EDCC 이벤트 세그먼트 시뮬레이션
WINDOW_SIZE = 400      # 2초 * 200Hz
WINDOW_STRIDE = 200    # 1초 * 200Hz
TRANSITION_MARGIN = 600  # ±3초 * 200Hz

def simulate_edcc_segments(events, signal_length):
    """이벤트 리스트에서 EDCC 세그먼트를 시뮬레이션"""
    segments = {'EYES_OPEN': 0, 'EYES_CLOSED': 0, 'EO_TO_EC': 0, 
                'EC_TO_EO': 0, 'PHOTIC': 0, 'OTHER': 0}
    
    # 관심 이벤트 필터링
    relevant_events = []
    for sample_idx, event_name in events:
        if event_name in ('Eyes Open', 'Eyes Closed') or event_name.startswith('Photic'):
            relevant_events.append((sample_idx, event_name))
    
    if not relevant_events:
        # 이벤트 없으면 전부 OTHER
        n_windows = max(0, (signal_length - WINDOW_SIZE) // WINDOW_STRIDE + 1)
        segments['OTHER'] = n_windows
        return segments
    
    # 각 구간 분류
    for i in range(len(relevant_events)):
        start_idx = relevant_events[i][0]
        event_name = relevant_events[i][1]
        
        if i + 1 < len(relevant_events):
            end_idx = relevant_events[i + 1][0]
        else:
            end_idx = signal_length
        
        # 구간 길이 (윈도우 수)
        seg_length = end_idx - start_idx
        n_windows = max(0, (seg_length - WINDOW_SIZE) // WINDOW_STRIDE + 1)
        
        if event_name == 'Eyes Open':
            segments['EYES_OPEN'] += n_windows
        elif event_name == 'Eyes Closed':
            segments['EYES_CLOSED'] += n_windows
        elif event_name.startswith('Photic On'):
            segments['PHOTIC'] += n_windows
        elif event_name == 'Photic Off':
            segments['OTHER'] += n_windows
    
    # 전환 구간 (EO↔EC 경계 ±3초)
    for i in range(len(relevant_events) - 1):
        curr_name = relevant_events[i][1]
        next_name = relevant_events[i + 1][1]
        
        if curr_name == 'Eyes Open' and next_name == 'Eyes Closed':
            segments['EO_TO_EC'] += max(1, (2 * TRANSITION_MARGIN - WINDOW_SIZE) // WINDOW_STRIDE + 1)
        elif curr_name == 'Eyes Closed' and next_name == 'Eyes Open':
            segments['EC_TO_EO'] += max(1, (2 * TRANSITION_MARGIN - WINDOW_SIZE) // WINDOW_STRIDE + 1)
    
    return segments

# 전체 데이터셋에 대해 시뮬레이션
seg_stats = []
for _, row in df_events.iterrows():
    segs = simulate_edcc_segments(row['events'], row['signal_length'])
    segs['serial'] = row['serial']
    segs['class_name'] = row['class_name']
    segs['total'] = sum(v for k, v in segs.items() if k not in ('serial', 'class_name'))
    seg_stats.append(segs)

df_seg_stats = pd.DataFrame(seg_stats)

# 세그먼트 타입별 윈도우 수 통계
print("=== EDCC 세그먼트 타입별 평균 윈도우 수 (per recording) ===")
seg_types = ['EYES_OPEN', 'EYES_CLOSED', 'EO_TO_EC', 'EC_TO_EO', 'PHOTIC', 'OTHER', 'total']
for seg_type in seg_types:
    vals = df_seg_stats[seg_type]
    print(f"  {seg_type:15s}: mean={vals.mean():6.1f}, std={vals.std():6.1f}, "
          f"min={vals.min():4.0f}, max={vals.max():6.0f}")

# 시각화
fig, ax = plt.subplots(figsize=(10, 5))
seg_means = df_seg_stats[seg_types[:-1]].mean()
bars = ax.bar(seg_means.index, seg_means.values, color=['#2ecc71', '#3498db', '#e67e22', '#e74c3c', '#9b59b6', '#95a5a6'])
ax.set_ylabel('Average number of windows')
ax.set_title('Average Windows per Segment Type per Recording')
for i, v in enumerate(seg_means.values):
    ax.text(i, v + 1, f'{v:.0f}', ha='center', fontweight='bold')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 7. 요약 및 EDCC 설계에 대한 시사점

**확인 사항**:
- [ ] EO/EC 이벤트가 없는 녹음이 있는지 → event_segmenter에서 OTHER 폴백 필요
- [ ] 구간 길이가 2초 윈도우보다 짧은 경우 비율 → 패딩 또는 스킵 전략
- [ ] 전환 횟수의 class별 차이 유무
- [ ] Photic stimulation 존재 비율 → PHOTIC 세그먼트 사용 가치
- [ ] 전환 구간 윈도우 수 vs 정적 구간 윈도우 수 비율 → 학습 시 샘플링 전략

**다음 단계**: Phase 1-3에서 주파수 대역별 파워 분석을 통해 class별 spectral signature를 확인합니다.